# Paper 001: inspect AAD on a Colab A100

Open this local notebook in VS Code, connect it through the official Google
Colab extension, and run the cells from top to bottom. The Python kernel and
filesystem are on the remote Colab machine. This ticket only inspects training
data; it does not train a model, run an experiment, or open test clips.

## 1. Confirm the remote A100 runtime

In [ ]:
import subprocess
import sys

import torch

gpu_report = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader",
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(gpu_report.stdout.strip())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Reconnect VS Code to the Colab A100.")
gpu_name = torch.cuda.get_device_name(0)
if "A100" not in gpu_name:
    raise RuntimeError(f"Expected an A100 runtime, found: {gpu_name}")
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA device:", gpu_name)

## 2. Reuse the remote code and dataset

The local notebook is only the interface. Source code must also exist on the
remote Colab filesystem. This cell reuses a checkout under
`/content/phd_research`, pulls only when it is clean, and clones only when it
is absent. It installs only missing non-PyTorch packages and reuses the
existing AAD KaggleHub cache; it never downloads the dataset.

In [ ]:
import importlib.util
import os
from pathlib import Path

REPOSITORY_URL = "https://github.com/sanelehlabisa/phd_research.git"
REPOSITORY_ROOT = Path("/content/phd_research")
IMPLEMENTATION_DIR = (
    REPOSITORY_ROOT
    / "papers"
    / "001-journal-abnormal-activity-recognition"
    / "implementation"
)

checkout_prepared = globals().get("_REMOTE_CHECKOUT_PREPARED", False)
if (REPOSITORY_ROOT / ".git").is_dir() and checkout_prepared:
    print("Reusing the checkout already prepared by this kernel.")
elif (REPOSITORY_ROOT / ".git").is_dir():
    git_status = subprocess.run(
        ["git", "status", "--porcelain"],
        cwd=REPOSITORY_ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    if git_status.stdout.strip():
        print("Reusing checkout; git pull skipped because it has local changes.")
    else:
        subprocess.run(
            ["git", "pull", "--ff-only"],
            cwd=REPOSITORY_ROOT,
            check=True,
        )
        print("Reused the existing clean checkout.")
elif REPOSITORY_ROOT.exists() and any(REPOSITORY_ROOT.iterdir()):
    raise RuntimeError(f"{REPOSITORY_ROOT} exists but is not a Git checkout")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY_ROOT)],
        check=True,
    )
    print("Created the remote code checkout.")
_REMOTE_CHECKOUT_PREPARED = True

if not IMPLEMENTATION_DIR.is_dir():
    raise FileNotFoundError(f"Implementation directory not found: {IMPLEMENTATION_DIR}")

required_packages = {
    "av": "av==16.1.0",
    "matplotlib": "matplotlib>=3.9,<4.0",
    "pandas": "pandas>=2.2,<3.0",
    "sklearn": "scikit-learn>=1.6,<2.0",
}
missing_packages = [
    package
    for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing_packages],
        check=True,
    )
    print("Installed missing packages:", missing_packages)
else:
    print("Required packages are already available.")

versions_dir = Path(
    "/root/.cache/kagglehub/datasets/sanelehlabisa/"
    "abnormal-activities-dataset/versions"
)
dataset_candidates = [
    path.resolve()
    for path in versions_dir.glob("*/abnormal-activities-dataset")
    if path.is_dir()
]
if not dataset_candidates:
    raise FileNotFoundError(
        f"AAD was not found under {versions_dir}. Add it before continuing."
    )
DATASET_DIR = max(
    dataset_candidates,
    key=lambda path: int(path.parent.name) if path.parent.name.isdigit() else -1,
)

os.chdir(IMPLEMENTATION_DIR)
if str(IMPLEMENTATION_DIR) not in sys.path:
    sys.path.insert(0, str(IMPLEMENTATION_DIR))
print("Implementation:", IMPLEMENTATION_DIR)
print("Dataset:", DATASET_DIR)

## 3. Load AAD and validate the committed split

The preview uses only reproducibly selected training clips. Split metadata is
validated and displayed, but no validation or test video is opened.

In [ ]:
import random

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from src.dataset import AHARDataset, VideoAugmentation, load_split_subsets
from src.utils import seed_everything

SEED = 42
SEQUENCE_LENGTH = 16
FRAME_SIZE = (32, 32)
PREVIEW_SAMPLES = 3
SPLIT_MANIFEST = (
    IMPLEMENTATION_DIR / "splits/abnormal-activities-dataset_seed42.json"
)

seed_everything(SEED)
dataset = AHARDataset(
    DATASET_DIR,
    sequence_length=SEQUENCE_LENGTH,
    frame_size=FRAME_SIZE,
)
train_subset, validation_subset, locked_test_subset, split_metadata = (
    load_split_subsets(dataset, SPLIT_MANIFEST, seed=SEED)
)
preview_indices = random.Random(SEED).sample(
    list(train_subset.indices),
    min(PREVIEW_SAMPLES, len(train_subset)),
)
preview_clips = []
for sample_index in preview_indices:
    video, label = dataset[sample_index]
    preview_clips.append((sample_index, video, label))

print("Validated manifest:", split_metadata["manifest_path"])
print("Loaded preview indices from the training split:", preview_indices)

## 4. Inspect dataset size, balance, and sample metadata

In [ ]:
first_video = preview_clips[0][1]
overview = pd.DataFrame(
    [
        ("Dataset path", str(DATASET_DIR)),
        ("Total clips", len(dataset)),
        ("Classes", dataset.num_classes),
        ("Train / validation / locked test", " / ".join(
            str(split_metadata["counts"][name])
            for name in ("train", "validation", "test")
        )),
        ("Preview tensor shape", str(tuple(first_video.shape))),
        ("Preview value range", f"{first_video.min():.3f} to {first_video.max():.3f}"),
        ("Split seed", split_metadata["seed"]),
    ],
    columns=["Item", "Value"],
)
display(overview)

class_rows = [
    {
        "Class": class_name,
        **split_metadata["class_counts"][class_name],
    }
    for class_name in dataset.class_names
]
class_counts = pd.DataFrame(class_rows)
display(class_counts)
axis = class_counts.set_index("Class")[["train", "validation", "test"]].plot(
    kind="bar",
    stacked=True,
    figsize=(11, 4),
    color=["#4c78a8", "#f2cf5b", "#e45756"],
)
axis.set_title("AAD class distribution from the committed split manifest")
axis.set_ylabel("Clips")
axis.legend(title="Split")
plt.tight_layout()
plt.show()

In [ ]:
sample_rows = []
for sample_index, video, label in preview_clips:
    source_path = dataset.samples[sample_index][0]
    sample_rows.append(
        {
            "Dataset index": sample_index,
            "Source": str(source_path.relative_to(DATASET_DIR)),
            "Class": dataset.class_names[label],
            "Tensor shape": str(tuple(video.shape)),
            "Minimum": round(float(video.min()), 3),
            "Maximum": round(float(video.max()), 3),
        }
    )
display(pd.DataFrame(sample_rows))

## 5. Compare real temporal frames before and after online augmentation

One parameter set is applied consistently to every frame in the augmented
clip. The augmentation preview has its own fixed seed so it is repeatable.

In [ ]:
sample_index, clean_video, sample_label = preview_clips[0]
with torch.random.fork_rng(devices=[]):
    torch.manual_seed(0)
    augmented_video = VideoAugmentation()(clean_video.clone())

frame_indices = [0, len(clean_video) // 2, len(clean_video) - 1]
figure, axes = plt.subplots(2, len(frame_indices), figsize=(11, 6))
for column, frame_index in enumerate(frame_indices):
    axes[0, column].imshow(clean_video[frame_index].permute(1, 2, 0).cpu())
    axes[0, column].set_title(f"Clean frame {frame_index}")
    axes[1, column].imshow(augmented_video[frame_index].permute(1, 2, 0).cpu())
    axes[1, column].set_title(f"Augmented frame {frame_index}")
    axes[0, column].axis("off")
    axes[1, column].axis("off")
source_name = dataset.samples[sample_index][0].name
class_name = dataset.class_names[sample_label]
figure.suptitle(f"{source_name} — {class_name}")
figure.tight_layout()
plt.show()
mean_change = (augmented_video - clean_video).abs().mean().item()
print(f"Mean absolute augmentation change: {mean_change:.4f}")

## 6. Next: inspect models

The A100, remote checkout, cached dataset, committed split, class balance, and
real training examples are now visible. No model, experiment, validation
ranking, checkpoint evaluation, or test clip has run. Ticket 023 adds model
inspection and one clearly labelled random-weight smoke prediction.